# 02｜ダンプから作品集合へ：文学作品の位置づけを考える

## なぜ大量の書誌データを使うのか？

一つの作品のEditionが100件あるとして、
それが多いのか少ないのかは、単独では判断できません。

そこで多数の作品の書誌情報を比較します。

このNotebookは、Open Libraryの大容量ダンプから
事前に作成した34,789件の固定データを利用します。
Colabで巨大なダンプそのものをダウンロードするわけではありません。

### この実習でやること

1. 作品集合のデータを読み込む。
2. 調べたい作品を選ぶ。
3. 同じ年代の作品群とEdition登録数を比較する。
4. Wikidataで作品の同定と関連情報を確認する。
5. OpenAlexの学術文献候補を確認し、解釈の限界を考える。

Open Library上の初刊年は、文学史上の初刊年と
一致するとは限りません。

OpenAlexの件数は暫定候補であり、
検証済みの研究論文数や文学的評価の順位ではありません。


## ① 固定された作品集合を読み込む


In [ ]:
# @title ① 研究用の作品集合を読み込む {display-mode: "form"}
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = "haruka0221/literary-metadata-workshop-2026"
REF = "main"
URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/workshop_search_v1.csv"

try:
    df = pd.read_csv(URL, dtype=str, keep_default_na=False, encoding="utf-8-sig")
except Exception as exc:
    raise RuntimeError("講習会用CSVがまだ公開されていないか、取得できません: " + URL) from exc

FLAG_LABELS = {
    "PROVISIONAL": "暫定（作品ID接続済み）",
    "PROVISIONAL_UNRESOLVED": "暫定・作品同定未確定",
    "NOT_AVAILABLE": "候補数を表示できない",
    "IDENTITY_HOLD": "作品同定保留",
}

required = {
    "ol_work_id", "ol_title", "ol_author_name",
    "ol_first_publish_year", "ol_edition_count",
    "openalex_display_candidate_count", "openalex_quality_flag",
    "openalex_count_unit_id", "wikidata_decision",
}
assert required <= set(df.columns), "CSVの列が想定と異なります"
assert len(df) == 34789, "CSVの行数が想定と異なります"

print(f"読み込み成功：{len(df):,} OLレコード")
display(df.head(3))


## ② 作品を選ぶ
01とは独立です。Work IDまたはタイトルで検索し、結果の**選択番号**を指定します。


In [ ]:
# @title ② 調べる作品を選ぶ {display-mode: "form"}
import pandas as pd
from IPython.display import display

# Work IDを知っていれば指定できます。空欄ならタイトル検索を利用。
WORK_ID = "" # @param {type:"string"}
TITLE_QUERY = "Ulysses" # @param {type:"string"}
AUTHOR_QUERY = "" # @param {type:"string"}
RESULT_NUMBER = 1 # @param {type:"integer", min:1, max:30}

if WORK_ID.strip():
    query = df[df["ol_work_id"].eq(WORK_ID.strip()) | df["current_ol_work_id"].eq(WORK_ID.strip())].copy()
else:
    query = df.copy()
    if TITLE_QUERY.strip():
        query = query[query["ol_title"].str.contains(TITLE_QUERY.strip(),case=False,regex=False,na=False)]
    if AUTHOR_QUERY.strip():
        query = query[query["ol_author_name"].str.contains(AUTHOR_QUERY.strip(),case=False,regex=False,na=False)]
    if not TITLE_QUERY.strip() and not AUTHOR_QUERY.strip():
        raise ValueError("Work ID、作品名、著者名のどれかを指定してください。")
query = query.copy()
query["_edition_num"] = pd.to_numeric(query["ol_edition_count"], errors="coerce")
query = query.sort_values("_edition_num",ascending=False)
print("固定集合の検索結果:",len(query), "（上位30件を表示）")
view = query[["ol_work_id","ol_title","ol_author_name","ol_first_publish_year","ol_edition_count"]].head(30).copy()
view.insert(0,"選択番号",range(1,len(view)+1))
display(view)
if query.empty:
    raise ValueError("固定集合に見つかりません。別の作品を入力してください。")
if RESULT_NUMBER < 1 or RESULT_NUMBER > len(query) or RESULT_NUMBER > 30:
    raise ValueError("表示された選択番号を指定してください。")
q = query.iloc[[RESULT_NUMBER-1]].copy()
chosen = q.iloc[0]
print("\n選択した作品:",chosen["ol_title"], "|", chosen["ol_work_id"])


## ③ 同じ年代の作品の中で、どの程度Editionが登録されているか
年はOpen Library由来です。文学史上の初刊年と一致しない場合があります。


In [ ]:
# @title ③ 同時代の作品群と比較する {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

all_years = pd.to_numeric(df["ol_first_publish_year"], errors="coerce")
all_editions = pd.to_numeric(df["ol_edition_count"],errors="coerce")
year = pd.to_numeric(pd.Series([chosen["ol_first_publish_year"]]),errors="coerce").iloc[0]
value = pd.to_numeric(pd.Series([chosen["ol_edition_count"]]),errors="coerce").iloc[0]
if pd.isna(year) or pd.isna(value):
    print("刊行年・Edition数に欠損があり、年代内比較はできません。")
else:
    start = int(year)//10*10
    peers = all_editions[all_years.between(start,start+9) & all_editions.notna()]
    print(f"Open Library記録の初刊年が{start}年代の歴史的Work：{len(peers):,}件")
    print("選択作品のEdition登録数:",int(value))
    if len(peers)>=30:
        above=int((peers>value).sum())
        ties=int((peers==value).sum())
        print(f"対象集合内でEdition登録数がこれより多いWork：{above:,}件")
        print(f"同件数のWork：{ties:,}件")
        print(f"この年代の登録Work中で上からおよそ{above+1:,}番目（同数あり／{len(peers):,}件）")
        fig,ax=plt.subplots(figsize=(8,4))
        ax.hist(np.log10(peers+1), bins=25)
        ax.axvline(np.log10(value+1), linestyle="--",linewidth=2,label="Selected Work")
        ax.set(xlabel="log10(Edition records + 1)",ylabel="Historical OL Work records",
               title=f"Edition records among OL Works: recorded {start}s")
        ax.legend();plt.tight_layout();plt.show()
    else:
        print("比較集合が30件未満のため、相対的な位置は表示しません。")
print("※比較対象は独自に抽出した34,789件であり、この年代の全小説ではありません。")
print("※OL年は実際の初刊年と異なる場合があり、登録Edition数は作品の評価を示しません。")


## ④ Wikidata・OpenAlexから何がわかるか
Wikidataは作品の同定・関連属性を調べる入口です。OpenAlexの件数は現時点では暫定候補であり、確定した学術論文数として順位づけしません。


In [ ]:
# @title ④ 外部データとの接続を確認する {display-mode: "form"}
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from IPython.display import display, Markdown

qid = chosen["wikidata_qid"].strip()
wd_status = chosen["wikidata_decision"]
print("=== 作品を外部の知識体系に接続する ===")
print("Wikidata同定:",wd_status)
if wd_status=="MATCH" and qid:
    display(Markdown(f"Wikidata作品項目：[{qid}](https://www.wikidata.org/wiki/{qid})"))
    try:
        req=Request(f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json",
                    headers={"User-Agent":"LiteraryMetadataWorkshop2026/0.1"})
        with urlopen(req,timeout=15) as resp:
            entity=json.load(resp)["entities"][qid]
        def label(x):
            info=entity.get(x,{})
            return info.get("ja",info.get("en",{})).get("value","（記述なし）")
        print("項目ラベル:",label("labels"))
        print("説明:",label("descriptions"))
        properties={"P50":"著者", "P136":"ジャンル", "P407":"作品の言語"}
        linked={}
        for pid,display_name in properties.items():
            ids=[]
            for statement in entity.get("claims",{}).get(pid,[]):
                val=statement.get("mainsnak",{}).get("datavalue",{}).get("value")
                if isinstance(val,dict) and val.get("id"):
                    ids.append(val["id"])
            linked[display_name]=list(dict.fromkeys(ids))[:8]
        lookups=list(dict.fromkeys(x for ids in linked.values() for x in ids))
        names={}
        if lookups:
            url="https://www.wikidata.org/w/api.php?"+urlencode({
                "action":"wbgetentities","ids":"|".join(lookups),
                "props":"labels","languages":"ja|en","format":"json",
            })
            req=Request(url,headers={"User-Agent":"LiteraryMetadataWorkshop2026/0.1"})
            with urlopen(req,timeout=15) as resp:
                entities=json.load(resp).get("entities",{})
            for other,desc in entities.items():
                labs=desc.get("labels",{})
                names[other]=labs.get("ja",labs.get("en",{})).get("value",other)
        for prop,ids in linked.items():
            print(prop+":", ", ".join(names.get(x,x) for x in ids) if ids else "記述なし")
    except Exception as exc:
        print("Wikidataの追加情報を取得できませんでした。上記リンクから確認できます。",type(exc).__name__)
    print("※Wikidataの記述は外部投稿に依存します。原典と照合して解釈してください。")
else:
    print("確定した作品QIDはありません。未接続は作品が存在しないという意味ではありません。")
print("\n=== 学術的な注目の候補 ===")
print("OpenAlex暫定候補数:",chosen["openalex_display_candidate_count"] or "記録なし")
print("計測状態:",chosen["openalex_quality_flag"])
print("候補の取得方法:",chosen["openalex_count_source"])
print("QA上の注意:",chosen["openalex_qa_note"] or "（記録なし）")
print("※OpenAlex候補数は照合検証済みの論文数ではありません。")
print("※文学的価値や学術的評価の直接指標・確定的ランキングとして使わないでください。")
print("\n視点：出版上の可視性（Edition）と学術的注目（候補文献）は同じでしょうか？")


## ⑤ 発展：作品集合全体を年代で絞り込み、Editionを比較する
ここからは②で選んだ作品とは独立した集合全体の分析です。


In [ ]:
# @title ⑤ 作品集合を年代で絞り込む {display-mode: "form"}
START_YEAR = 1880 # @param {type:"integer"}
END_YEAR = 1900 # @param {type:"integer"}
years = pd.to_numeric(df["ol_first_publish_year"], errors="coerce")
period = df[years.between(START_YEAR, END_YEAR)].copy()
print(f"この読み込みデータ内で{START_YEAR}–{END_YEAR}年に該当：{len(period):,} OLレコード")
display(period[["ol_title","ol_author_name","ol_first_publish_year"]].head(25))

In [ ]:
# @title ⑥ Edition登録数を比較する {display-mode: "form"}
PLOT_TOP = 15 # @param {type:"integer", min:5, max:40}
x = period.copy()
x["edition_numeric"] = pd.to_numeric(x["ol_edition_count"], errors="coerce")
x = x.dropna(subset=["edition_numeric"]).nlargest(PLOT_TOP,"edition_numeric")
if x.empty: print("このデータにはEdition数が記録されていません。")
else:
    display(x[["ol_title","ol_author_name","ol_edition_count"]])
    ax=x.sort_values("edition_numeric").plot.barh(x="ol_title",y="edition_numeric",legend=False,figsize=(9,max(4,len(x)*.38)))
    ax.set(xlabel="Registered OL Edition records",ylabel="OL title"); plt.tight_layout(); plt.show()

## 考えてみる
- 同年代でEdition登録が多いことは、どんな出版史的現象を反映するでしょうか。
- Wikidataの記述・同定をどのように検証できますか。
- OpenAlexで学術的注目を比較するには、なぜ作品の同定と候補論文の精度検証が必要でしょうか。
- 読者受容・教育利用を調べるには、ほかにどのようなデータが必要でしょうか。
